In [ ]:
# 加载模块
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

In [ ]:
# Load module
data = pd.read_csv('01_12_clusters_data\\12.csv')
data

In [ ]:
# Set input features and output features
y = data.n
x = data[['LON', 'LAT', 'AREA', 'ELE', 'SLP', 'RIVER', 'CLAY', 'SAND', 'SILT', 'CTI', 'SW', 'SPEED', 'LAI', 'SRAD', 'CO2']].select_dtypes(exclude=['object'])

In [ ]:
# Randomly split the training set and test set in a 4:6 ratio
x_train, x_test, y_train, y_test = train_test_split(x.values, y.values, test_size=0.4, random_state=42)

In [ ]:
#Enter the best hyperparameters for each cluster, see comments
opt_xgb=XGBRegressor(colsample_bytree=1.0, learning_rate=0.1, max_depth=12, n_estimators=200, reg_alpha=0.1, reg_lambda=0, subsample=0.8)
opt_xgb.fit(x_train, y_train)

In [ ]:
# Use the model to make predictions on the training set and test set
y_train_pred = opt_xgb.predict(x_train)
y_test_pred = opt_xgb.predict(x_test)

In [ ]:
# Calculate evaluation metrics: RMSE, R²
# RMSE
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
rmse_train = rmse(y_train, y_train_pred)
rmse_test = rmse(y_test, y_test_pred)

# R²
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# NSE (Nash‑Sutcliffe Efficiency)
def NSE(y_pred, y_true):
    ave_obs = sum(y_true)/len(y_true)
    Numerator = sum((y_true-y_pred)**2)
    Denominator = sum((y_true-ave_obs)**2)
    return 1 - Numerator/Denominator
nse_train = NSE(y_train, y_train_pred)
nse_test = NSE(y_test, y_test_pred)

print("Training set evaluation:")
print(f"RMSE: {rmse_train:.4f}, R²: {r2_train:.4f}, NSE: {nse_train:.4f}")
print("Test set evaluation:")
print(f"RMSE: {rmse_test:.4f}, R²: {r2_test:.4f}, NSE: {nse_test:.4f}")

In [ ]:
# Plot training set fitting graph
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.7)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel('Observed_n')
plt.ylabel('Prediced_n')
plt.title('train set')
plt.text(0.05, 0.95, f"RMSE: {rmse_train:.4f}\nR²: {r2_train:.4f}\nNSE: {nse_train:.4f}", 
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Observed_n')
plt.ylabel('Prediced_n')
plt.title('test set')
plt.text(0.05, 0.95, f"RMSE: {rmse_test:.4f}\nR²: {r2_test:.4f}\nNSE: {nse_test:.4f}", 
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

In [ ]:
# Prediction
new_data_path = r'01_12_clusters_data\\12.csv' 
new_data = pd.read_csv(new_data_path)

required_columns = ['LON', 'LAT', 'AREA', 'ELE', 'SLP', 'RIVER', 'CLAY', 'SAND', 'SILT', 'CTI', 'SW', 'SPEED', 'LAI', 'SRAD', 'CO2']

# Extract features
X_new = new_data[required_columns]

# Predict using the trained XGBoost model
y_new_pred = opt_xgb.predict(X_new.values)

new_data['n'] = y_new_pred

output_path = r'03_12_clusters_p\\12_p.csv'
new_data.to_csv(output_path, index=False)

print(f"Prediction results have been saved to file: {output_path}")